In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
from sklearn.utils import resample
from data_loader import load_merged_data




# # Experiment 6 — Bootstrap Confidence Interval on Biomarker Model Accuracy
# **Goal:** Quantify uncertainty on the reported holdout accuracy using 1,000 bootstrap resamples.




In [ ]:
print("Loading Models and Data...")
data_dir = "../data"
merged, dx_col = load_merged_data(data_dir)
merged = merged.dropna(subset=[dx_col])
valid_dx = ['CN', 'MCI', 'Dementia']
merged[dx_col] = merged[dx_col].replace({'AD': 'Dementia'})
merged = merged[merged[dx_col].isin(valid_dx)]

alele_model_dir = "../ml/alele_model/model_7f/content/model_7f"
xgb_model = joblib.load(os.path.join(alele_model_dir, "xgb_model.joblib"))
rf_model  = joblib.load(os.path.join(alele_model_dir, "rf_model.joblib"))
nn_model  = joblib.load(os.path.join(alele_model_dir, "nn_model.joblib"))
blend_weights = joblib.load(os.path.join(alele_model_dir, "blend_weights_optimized.joblib"))
scaler = joblib.load(os.path.join(alele_model_dir, "scaler.joblib"))
le_dx  = joblib.load(os.path.join(alele_model_dir, "label_encoder.joblib"))
trained_columns = joblib.load(os.path.join(alele_model_dir, "trained_columns.joblib"))

# Prepare original features
features = ['ABETA', 'TAU', 'MMSE', 'APVOLUME']
genotype_cols = [c for c in merged.columns if 'APOE' in c.upper()]
new_patient = merged[features].copy()
new_patient['GENOTYPE'] = merged[genotype_cols[0]]
new_patient['ABETA_TAU_RATIO'] = new_patient['ABETA'] / new_patient['TAU']
new_patient['MMSE_TAU_INTERACT'] = new_patient['MMSE'] * new_patient['TAU']

encoded = pd.get_dummies(new_patient)
aligned_input = pd.DataFrame(np.zeros((len(merged), len(trained_columns))), columns=trained_columns)
for c in encoded.columns:
    if c in aligned_input.columns:
        aligned_input[c] = encoded[c].values

aligned_input_scaled = scaler.transform(aligned_input.fillna(0))

y = le_dx.transform(merged[dx_col])

# Recreate the 80/20 holdout split used in training
X_train, X_test, y_train, y_test = train_test_split(aligned_input_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Holdout set size: {len(y_test)}")




# ## Run Bootstrap




In [ ]:
try: n_xgb = xgb_model.n_features_in_
except: n_xgb = xgb_model.get_booster().num_features()

p_xgb = xgb_model.predict_proba(X_test[:, :n_xgb])
p_rf  = rf_model.predict_proba(X_test[:, :rf_model.n_features_in_])
p_nn  = nn_model.predict_proba(X_test[:, :nn_model.n_features_in_])

blend_probs = (blend_weights['w_xgb'] * p_xgb + blend_weights['w_rf'] * p_rf + blend_weights['w_nn'] * p_nn)
preds = np.argmax(blend_probs, axis=1)

n_iterations = 1000
boot_acc = []
boot_f1 = []

np.random.seed(42)
for i in range(n_iterations):
    # Resample indices with replacement
    indices = resample(np.arange(len(y_test)), replace=True)
    y_true_boot = y_test[indices]
    y_pred_boot = preds[indices]
    
    acc = accuracy_score(y_true_boot, y_pred_boot)
    _, _, f1, _ = precision_recall_fscore_support(y_true_boot, y_pred_boot, average='weighted', zero_division=0)
    
    boot_acc.append(acc)
    boot_f1.append(f1)

ci_lower_acc = np.percentile(boot_acc, 2.5)
ci_upper_acc = np.percentile(boot_acc, 97.5)
mean_acc = np.mean(boot_acc)

ci_lower_f1 = np.percentile(boot_f1, 2.5)
ci_upper_f1 = np.percentile(boot_f1, 97.5)
mean_f1 = np.mean(boot_f1)




# ## Plotting




In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(boot_acc, bins=30, kde=True, color='skyblue')
plt.axvline(mean_acc, color='red', linestyle='--', label=f'Mean: {mean_acc:.3f}')
plt.axvline(ci_lower_acc, color='black', linestyle=':', label=f'95% CI: [{ci_lower_acc:.3f}, {ci_upper_acc:.3f}]')
plt.axvline(ci_upper_acc, color='black', linestyle=':')
plt.title('Bootstrap Accuracy Distribution (Biomarker Ensemble)')
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.savefig('figures/06_bootstrap_acc_hist.png', dpi=300)
plt.close()

# -------------------------
# Clinical CatBoost Bootstrap
# -------------------------
print("Bootstrapping Clinical Model...")
catboost_model = joblib.load("../ml/base_model/catboost_alzheimer_model.pkl")
minmax = joblib.load("../ml/base_model/minmax.pkl")
standard = joblib.load("../ml/base_model/standard.pkl")
df_clin_raw = pd.read_csv('../data/alzheimers_disease_data.csv')
y_clin = df_clin_raw['Diagnosis'].values
cb_cols = ['Age', 'BMI', 'AlcoholConsumption', 'PhysicalActivity', 'DietQuality',
    'SleepQuality', 'SystolicBP', 'DiastolicBP', 'CholesterolTotal',
    'CholesterolLDL', 'CholesterolHDL', 'CholesterolTriglycerides',
    'MMSE', 'FunctionalAssessment', 'ADL']
X_clin = df_clin_raw[['PatientID', 'Age', 'Gender', 'Ethnicity', 'EducationLevel', 'BMI', 'Smoking', 'AlcoholConsumption', 'PhysicalActivity', 'DietQuality', 'SleepQuality', 'FamilyHistoryAlzheimers', 'CardiovascularDisease', 'Diabetes', 'Depression', 'HeadInjury', 'Hypertension', 'SystolicBP', 'DiastolicBP', 'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL', 'CholesterolTriglycerides', 'MMSE', 'FunctionalAssessment', 'MemoryComplaints', 'BehavioralProblems', 'ADL', 'Confusion', 'Disorientation', 'PersonalityChanges', 'DifficultyCompletingTasks', 'Forgetfulness', 'DoctorInCharge']].copy()

X_clin[cb_cols] = minmax.transform(X_clin[cb_cols])
X_clin[cb_cols] = standard.transform(X_clin[cb_cols])

# Handle XXXConfid string issue in DoctorInCharge
X_clin.replace('XXXConfid', 0, inplace=True)
for c in X_clin.columns:
    if X_clin[c].dtype == 'object':
        try: X_clin[c] = X_clin[c].astype(float)
        except: pass

_, X_test_clin, _, y_test_clin = train_test_split(X_clin, y_clin, test_size=0.2, random_state=42, stratify=y_clin)
preds_clin = catboost_model.predict(X_test_clin)

boot_acc_clin, boot_f1_clin = [], []
np.random.seed(42)
for i in range(n_iterations):
    indices = resample(np.arange(len(y_test_clin)), replace=True)
    y_true_boot = y_test_clin[indices]
    y_pred_boot = preds_clin[indices]
    boot_acc_clin.append(accuracy_score(y_true_boot, y_pred_boot))
    boot_f1_clin.append(f1_score(y_true_boot, y_pred_boot, average='weighted', zero_division=0))

mean_acc_clin = np.mean(boot_acc_clin)
ci_lower_acc_clin = np.percentile(boot_acc_clin, 2.5)
ci_upper_acc_clin = np.percentile(boot_acc_clin, 97.5)
mean_f1_clin = np.mean(boot_f1_clin)
ci_lower_f1_clin = np.percentile(boot_f1_clin, 2.5)
ci_upper_f1_clin = np.percentile(boot_f1_clin, 97.5)

with open('results/06_bootstrap_results.txt', 'w') as f:
    f.write("--- Biomarker Ensemble ---\n")
    f.write(f"Accuracy: {mean_acc:.3f} [95% CI: {ci_lower_acc:.3f} - {ci_upper_acc:.3f}]\n")
    f.write(f"F1 Score: {mean_f1:.3f} [95% CI: {ci_lower_f1:.3f} - {ci_upper_f1:.3f}]\n\n")
    f.write("--- Clinical CatBoost ---\n")
    f.write(f"Accuracy: {mean_acc_clin:.3f} [95% CI: {ci_lower_acc_clin:.3f} - {ci_upper_acc_clin:.3f}]\n")
    f.write(f"F1 Score: {mean_f1_clin:.3f} [95% CI: {ci_lower_f1_clin:.3f} - {ci_upper_f1_clin:.3f}]\n")




# # Summary
# 
# **Headline Findings:**
# - **Clinical CatBoost:** Test accuracy is {mean_acc_clin:.3f} [95% CI: {ci_lower_acc_clin:.3f}–{ci_upper_acc_clin:.3f}].
# - **Clinical CatBoost:** Weighted F1 score is {mean_f1_clin:.3f} [95% CI: {ci_lower_f1_clin:.3f}–{ci_upper_f1_clin:.3f}].
# - **Biomarker Ensemble:** Weighted F1 score is {mean_f1:.3f} [95% CI: {ci_lower_f1:.3f}–{ci_upper_f1:.3f}].
# - The robust clinical CI confirms high reliability when complete patient history is available.

